In [20]:
import pandas as pd
#%pip install -U openai pydantic

import os
from dotenv import load_dotenv
from openai import OpenAI


from typing import Literal
from pydantic import BaseModel

from pathlib import Path

import json
from datetime import datetime, timezone

In [7]:
#%pip install -U openai pydantic python-dotenv

In [8]:
df_reviews = pd.read_csv("../data/processed/santa_barbara_data.csv")

df_reviews.head()

/var/folders/1y/mz2h8lvj4sn_p_w4hfwq7m6w0000gn/T/ipykernel_53701/526265633.py:1: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or set low_memory=False.
  df_reviews = pd.read_csv("../data/processed/santa_barbara_data.csv")


,review_id,user_id,business_id,stars_x,useful,funny,cool,text,date,name,...,Wednesday_start,Wednesday_end,Thursday_start,Thursday_end,Friday_start,Friday_end,Saturday_start,Saturday_end,Sunday_start,Sunday_end
0,L0jv8c2FbpWSlfNC6bbUEA,bFPdtzu11Oi0f92EAcjqmg,IDtLPgUrqorrpqSLdfMhZQ,5,0,0,0,What a great addition to the Funk Zone! Grab ...,2016-10-13 22:50:47,Helena Avenue Bakery,...,8:0,14:0,8:0,14:0,8:0,14:0,8:0,14:0,8:0,14:0
1,SpmKsFnwg35PTgCVujOOJA,RiQWbgwR-asiU_mA7s8gpg,pUVRCYL8rT4I5Ry5FYkRsA,5,2,0,1,Sweet spot in the Waterline. Came for some liv...,2016-12-19 22:58:19,Topa Topa Brewing Company,...,12:0,22:0,12:0,22:0,12:0,23:0,12:0,23:0,12:0,20:0
2,Gczff1Qxg4WVmlBLPrZwzQ,cpp89UW0imv45y3a_O444w,34Eqv8jXgxg_EEwcsNgeeg,5,0,0,0,We absolutely love everything we have tried he...,2014-05-25 22:33:02,Backyard Bowls,...,8:0,17:0,8:0,17:0,8:0,17:0,8:0,17:0,8:0,17:0
3,bi6GaeWDGceGv62lXTIKQA,RgtbLaiU22zqaCk20HgbiQ,bjhCtlYHrkgA5Ku8l-rB3g,1,1,0,0,Very disappointed. We went to eat at 2:15 on ...,2015-01-04 02:26:46,Our Daily Bread,...,6:30,15:30,6:30,15:30,6:30,15:30,6:30,15:30,6:30,16:0
4,H4mSxGeKTv4VFcc9YUmtpQ,oGV84fsuRbLQRRQ6ZZC1dA,-kY_HDP7IMvGl-kBIZVU4A,4,0,0,0,The espresso here is excellent. My favorite pl...,2016-07-14 19:43:42,Dune Coffee Roasters - Anacapa,...,6:0,17:0,6:0,17:0,6:0,17:0,6:0,17:0,7:0,17:0


In [9]:
blank_text = df_reviews["text"].fillna("").str.strip().eq("")

print("Reviews with blank text:", blank_text.sum())

Reviews with blank text: 0


In [10]:
# Keep only the fields needed for labeling.
labeling_data = df_reviews[
    ["review_id", "business_id", "text"]
].copy()

# Shuffle reproducibly, then select nonoverlapping samples.
shuffled = labeling_data.sample(frac=1, random_state=42)

df_pilot = shuffled.iloc[:100].copy()
df_holdout = shuffled.iloc[100:300].copy()

assert len(labeling_data) >= 300
assert set(df_pilot["review_id"]).isdisjoint(df_holdout["review_id"])

print("Pilot reviews:", len(df_pilot))
print("Holdout reviews:", len(df_holdout))

Pilot reviews: 100
Holdout reviews: 200


In [11]:
qualifiers = [
    "good_wifi",
    "easy_parking",
    "quiet_atmosphere",
    "enough_seating",
    "outlets_available",
    "work_friendly",
    "good_drinks",
    "fun_atmosphere",
    "social_setting"]

Actuallly starting to work wil LLM now

In [12]:
load_dotenv("../src/.env")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("API key not found. Check your .env file.")

client = OpenAI()

In [13]:
#defining output labels

class ReviewLabels(BaseModel):
    good_wifi: Literal[-1, 0, 1]
    easy_parking: Literal[-1, 0, 1]
    quiet_atmosphere: Literal[-1, 0, 1]
    enough_seating: Literal[-1, 0, 1]
    outlets_available: Literal[-1, 0, 1]
    work_friendly: Literal[-1, 0, 1]
    good_drinks: Literal[-1, 0, 1]
    fun_atmosphere: Literal[-1, 0, 1]
    social_setting: Literal[-1, 0, 1]

In [16]:
prompt_path = Path("../src/promt.txt")

PROMPT = prompt_path.read_text(encoding="utf-8").strip()
PROMPT_VERSION = "pilot_v1"

if not PROMPT:
    raise ValueError("The prompt file is empty.")

print("Prompt loaded.")

Prompt loaded.


In [17]:
# Pick the first review from your pilot sample.
review = df_pilot.iloc[0]

print("REVIEW:\n")
print(review["text"])

REVIEW:

Iced Chai was delicious and just what I needed on this beautiful day. After getting to the store next door and realizing I left my sunglasses somewhere and of course I retraced my steps. They led me back to The coffee shop. Imagine my surprise when I walk and and the barista is wearing my Ray Bans and asking her counter part if they look good on her. I walk in ask if I left my Ray Bans and sheepishly her coworker hands them over. Pretty sad to think they wouldn't have at least put them in lost and found for a few hours let alone minutes... :( sad because I am "that girl" who never has her things put in lost and found. Happy I walked in at the right moment and got them back:)


In [19]:

# Send your prompt and the review text to the model.
response = client.responses.parse(
    model="gpt-4.1-mini",
    input=[
        {"role": "system", "content": PROMPT},
        {"role": "user", "content": review["text"]},
    ],
    text_format=ReviewLabels,
    store=False,
)

if response.output_parsed is None:
    raise RuntimeError("The model did not return labels. Inspect response.")

# Display the nine predicted labels.
labels = response.output_parsed.model_dump()

pd.DataFrame(
    labels.items(),
    columns=["qualifier", "llm_label"],
)

,qualifier,llm_label
0,good_wifi,0
1,easy_parking,0
2,quiet_atmosphere,0
3,enough_seating,0
4,outlets_available,0
5,work_friendly,0
6,good_drinks,1
7,fun_atmosphere,-1
8,social_setting,0


In [22]:

MODEL = "gpt-4.1-mini"
PROMPT_VERSION = "pilot_v1"

output_path = Path(
    "/Users/rejaimes/repos/coffee-project/data/processed"
) / f"pilot_llm_labels_{PROMPT_VERSION}.jsonl"
# Load any previously saved predictions.
results = {}

if output_path.exists():
    for line in output_path.read_text().splitlines():
        if line.strip():
            record = json.loads(line)
            results[record["review_id"]] = record


def save_prediction(row, parsed_labels, model):
    record = {
        "review_id": row["review_id"],
        "business_id": row["business_id"],
        **parsed_labels,
        "label_model": model,
        "prompt_version": PROMPT_VERSION,
        "label_date": datetime.now(timezone.utc).isoformat(),
    }

    with output_path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record) + "\n")

    results[record["review_id"]] = record


# Reuse the successful first response.
# Run this before changing your prompt or the `review` variable.
if review["review_id"] not in results:
    if response.output_parsed is None:
        raise RuntimeError("The first response has no parsed labels.")

    save_prediction(
        review,
        response.output_parsed.model_dump(),
        response.model,
    )

# Label the remaining reviews.
for row in df_pilot.to_dict("records"):
    if row["review_id"] in results:
        continue

    response = client.responses.parse(
        model=MODEL,
        input=[
            {"role": "system", "content": PROMPT},
            {"role": "user", "content": row["text"]},
        ],
        text_format=ReviewLabels,
        store=False,
    )

    if response.output_parsed is None:
        raise RuntimeError(
            f"No labels returned for {row['review_id']}. "
            "Previously completed predictions are saved."
        )

    save_prediction(
        row,
        response.output_parsed.model_dump(),
        response.model,
    )

    print(f"Saved {len(results)}/{len(df_pilot)} reviews")

df_llm = pd.DataFrame(results.values())

Saved 2/100 reviews
Saved 3/100 reviews
Saved 4/100 reviews
Saved 5/100 reviews
Saved 6/100 reviews
Saved 7/100 reviews
Saved 8/100 reviews
Saved 9/100 reviews
Saved 10/100 reviews
Saved 11/100 reviews
Saved 12/100 reviews
Saved 13/100 reviews
Saved 14/100 reviews
Saved 15/100 reviews
Saved 16/100 reviews
Saved 17/100 reviews
Saved 18/100 reviews
Saved 19/100 reviews
Saved 20/100 reviews
Saved 21/100 reviews
Saved 22/100 reviews
Saved 23/100 reviews
Saved 24/100 reviews
Saved 25/100 reviews
Saved 26/100 reviews
Saved 27/100 reviews
Saved 28/100 reviews
Saved 29/100 reviews
Saved 30/100 reviews
Saved 31/100 reviews
Saved 32/100 reviews
Saved 33/100 reviews
Saved 34/100 reviews
Saved 35/100 reviews
Saved 36/100 reviews
Saved 37/100 reviews
Saved 38/100 reviews
Saved 39/100 reviews
Saved 40/100 reviews
Saved 41/100 reviews
Saved 42/100 reviews
Saved 43/100 reviews
Saved 44/100 reviews
Saved 45/100 reviews
Saved 46/100 reviews
Saved 47/100 reviews
Saved 48/100 reviews
Saved 49/100 reviews


In [23]:
df_human = pd.read_csv( "../data/processed/pilot_human_labels.csv")
df_llm = pd.read_json("../data/processed/pilot_llm_labels_pilot_v1.jsonl",
    lines=True,
)

In [24]:
comparison = df_human[
    ["review_id", "text"] + qualifiers
].merge(
    df_llm[["review_id"] + qualifiers],
    on="review_id",
    suffixes=("_human", "_llm"),
    validate="one_to_one",
)

In [25]:
summary = []

for qualifier in qualifiers:
    matches = (
        comparison[f"{qualifier}_human"]
        == comparison[f"{qualifier}_llm"]
    )

    summary.append({
        "qualifier": qualifier,
        "matches": int(matches.sum()),
        "disagreements": int((~matches).sum()),
        "agreement_pct": round(matches.mean() * 100, 1),
    })

df_agreement = pd.DataFrame(summary)
df_agreement.sort_values("disagreements", ascending=False)

,qualifier,matches,disagreements,agreement_pct
7,fun_atmosphere,67,33,67.0
6,good_drinks,73,27,73.0
8,social_setting,77,23,77.0
2,quiet_atmosphere,91,9,91.0
5,work_friendly,92,8,92.0
3,enough_seating,94,6,94.0
1,easy_parking,97,3,97.0
0,good_wifi,100,0,100.0
4,outlets_available,100,0,100.0


In [26]:
disagreements = []

for qualifier in qualifiers:
    different = (
        comparison[f"{qualifier}_human"]
        != comparison[f"{qualifier}_llm"]
    )

    for _, row in comparison.loc[different].iterrows():
        disagreements.append({
            "review_id": row["review_id"],
            "qualifier": qualifier,
            "human_label": row[f"{qualifier}_human"],
            "llm_label": row[f"{qualifier}_llm"],
            "text": row["text"],
        })

df_disagreements = pd.DataFrame(
    disagreements,
    columns=["review_id", "qualifier", "human_label", "llm_label", "text"],
)

df_disagreements.to_csv("../data/processed/pilot_disagreements_v1.csv",
    index=False,
)

df_disagreements.head(10)

,review_id,qualifier,human_label,llm_label,text
0,BC5KqFNVYF6R9Up6XyYVHA,easy_parking,0,-1,I wish i could give this place 5 stars but the...
1,zQgfByTL2INJXX2nszuAgw,easy_parking,0,-1,Hipster Coffee. Handlebars is a cute and trend...
2,LQJD_QJOgpJrpkxAixdN8A,easy_parking,0,1,Oat Bakery is so convenient with easy walk up ...
3,SzswJQTBjs2CrqjYZwzVew,quiet_atmosphere,0,1,"HUGE FAN. I currently live in Chicago, and do..."
4,azlzlJAXwj_ieXkvison-A,quiet_atmosphere,0,-1,we came here with a big wedding party the day ...
5,9WQC7BFh5TA90H9iJgwg0Q,quiet_atmosphere,0,-1,"Her salad was good, but nothing special my wif..."
6,uSAYpv-_X09sQvIbjh7oZA,quiet_atmosphere,0,-1,I legit come here many times a week. It does g...
7,yD_G_X0BwRoiUG52X_V3xQ,quiet_atmosphere,0,-1,A hip place to hang out. Lots of people comin...
8,nt3cJbXFff9BxcFUjZP8-A,quiet_atmosphere,0,1,Family business .... Smiles and a wonderfully ...
9,K0iwbVqA3_ViGGL7pxiSaQ,quiet_atmosphere,0,-1,This place was a great place to go on a saturd...


In [31]:
#from pathlib import Path
PROMPT_VERSION = "pilot_v2"

PROMPT = Path("../src/prompt_v2.txt").read_text(encoding="utf-8").strip()

assert PROMPT, "Prompt file is empty."

In [33]:
project_dir = Path("/Users/rejaimes/repos/coffee-project")

PROMPT = (
    project_dir / "src" / "prompt_v2.txt"
).read_text(encoding="utf-8").strip()

PROMPT_VERSION = "pilot_v2"

assert PROMPT, "The prompt file is empty."
print("Version 2 prompt loaded.")

Version 2 prompt loaded.


In [34]:
MODEL = "gpt-4.1-mini"

output_path = (
    project_dir / "data" / "processed"
    / f"pilot_llm_labels_{PROMPT_VERSION}.jsonl"
)

output_path.parent.mkdir(parents=True, exist_ok=True)

assert len(df_pilot) == 100
assert df_pilot["review_id"].is_unique

# Load any version 2 results already completed.
results = {}

if output_path.exists():
    for line in output_path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            record = json.loads(line)
            results[record["review_id"]] = record

# Send each unfinished review using the revised prompt.
for row in df_pilot.to_dict("records"):
    if row["review_id"] in results:
        continue

    response = client.responses.parse(
        model=MODEL,
        input=[
            {"role": "system", "content": PROMPT},
            {"role": "user", "content": row["text"]},
        ],
        text_format=ReviewLabels,
        store=False,
    )

    if response.output_parsed is None:
        raise RuntimeError(
            f"No labels returned for {row['review_id']}. "
            "Earlier results are saved."
        )

    record = {
        "review_id": row["review_id"],
        "business_id": row["business_id"],
        **response.output_parsed.model_dump(),
        "label_model": response.model,
        "prompt_version": PROMPT_VERSION,
        "label_date": datetime.now(timezone.utc).isoformat(),
    }

    with output_path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record) + "\n")

    results[row["review_id"]] = record
    print(f"Saved {len(results)}/100 reviews")

df_llm_v2 = pd.DataFrame(results.values())

print("Version 2 run complete.")

Saved 1/100 reviews
Saved 2/100 reviews
Saved 3/100 reviews
Saved 4/100 reviews
Saved 5/100 reviews
Saved 6/100 reviews
Saved 7/100 reviews
Saved 8/100 reviews
Saved 9/100 reviews
Saved 10/100 reviews
Saved 11/100 reviews
Saved 12/100 reviews
Saved 13/100 reviews
Saved 14/100 reviews
Saved 15/100 reviews
Saved 16/100 reviews
Saved 17/100 reviews
Saved 18/100 reviews
Saved 19/100 reviews
Saved 20/100 reviews
Saved 21/100 reviews
Saved 22/100 reviews
Saved 23/100 reviews
Saved 24/100 reviews
Saved 25/100 reviews
Saved 26/100 reviews
Saved 27/100 reviews
Saved 28/100 reviews
Saved 29/100 reviews
Saved 30/100 reviews
Saved 31/100 reviews
Saved 32/100 reviews
Saved 33/100 reviews
Saved 34/100 reviews
Saved 35/100 reviews
Saved 36/100 reviews
Saved 37/100 reviews
Saved 38/100 reviews
Saved 39/100 reviews
Saved 40/100 reviews
Saved 41/100 reviews
Saved 42/100 reviews
Saved 43/100 reviews
Saved 44/100 reviews
Saved 45/100 reviews
Saved 46/100 reviews
Saved 47/100 reviews
Saved 48/100 reviews
S

In [35]:
df_disagreements_v2 = pd.read_csv(
    "/Users/rejaimes/repos/coffee-project/data/processed/"
    "pilot_disagreements_v2_reviewed_2026-10-22.csv"
)

df_disagreements_v2.head(10)

,review_id,qualifier,human_label,llm_v1_label,llm_label,text
0,gMtuPeIBkM7O1Rn3R6Ksbw,quiet_atmosphere,0,0,-1,I live two blocks from Daily Grind and I eat i...
1,azlzlJAXwj_ieXkvison-A,quiet_atmosphere,-1,-1,0,we came here with a big wedding party the day ...
2,9WQC7BFh5TA90H9iJgwg0Q,quiet_atmosphere,0,-1,-1,"Her salad was good, but nothing special my wif..."
3,yD_G_X0BwRoiUG52X_V3xQ,quiet_atmosphere,-1,-1,0,A hip place to hang out. Lots of people comin...
4,3JYWLBb8jExWufjgjJFmYw,quiet_atmosphere,-1,0,0,"My Daughter is a regular here and loves it, so..."
5,sV19g2ggql1tlZeaPMWgnw,quiet_atmosphere,0,-1,-1,I'm glad this is something I tried and it was ...
6,JxT7pEFiFNmo1HItkBes6Q,enough_seating,0,0,1,Here in SB from Portland OR for a girls' weeke...
7,azlzlJAXwj_ieXkvison-A,enough_seating,-1,1,1,we came here with a big wedding party the day ...
8,tx_ykeK9xSDIVQ-EsrhLag,enough_seating,0,0,1,Fabulous and hip Santa Barbara coffee shop: in...
9,1BjjfV4eEPs2_A5MyyRKVw,enough_seating,1,1,0,Nice coffee shop with a local vibe. I really ...
